In [1]:
import os
from shared_lib.local import LOCAL_ENV, LOCAL_RUN
from shared_lib.spark import (
    get_spark_session,
)
spark = get_spark_session(
    app_name="Aggtrades Ingestion Job", master=True, jars=True, local_run=LOCAL_RUN, minio=LOCAL_ENV, hive=LOCAL_ENV
)

data_lake_bucket = os.getenv("DATA_LAKE_BUCKET", "crypto-data-lake")
transform_db = os.getenv("TRANSFORM_DB", "crypto_transform")


:: loading settings :: url = jar:file:/Users/anhtu/.pyenv/versions/3.11.11/envs/spark/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/anhtu/.ivy2/cache
The jars for the packages stored in: /Users/anhtu/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9743a3e5-30a9-44c5-b268-12f229a245b4;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.7.1 in central
	found org.apache.iceberg#iceberg-aws-bundle;1.7.1 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 103ms :: artifacts dl 3ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.apache.iceberg#iceberg-aws-bundle;1.7.1 from cent

In [ ]:
ema_signals_df = spark.sql(f"""
with cte as (
    select 
        *,
        lag(ema7) over(partition by symbol, interval order by open_time) as ema7_prev,
        lag(ema20) over(partition by symbol, interval order by open_time) as ema20_prev
    from {transform_db}.kline_features
), ema_cross_signals as (
    select
        symbol,
        interval,
        open_time,
        date,
        close_price,
        case
            when ema7_prev is null or ema20_prev is null then 'none'
            when ema7_prev <= ema20_prev and ema7 > ema20 then 'ema_cross_up'
            when ema7_prev >= ema20_prev and ema7 < ema20 then 'ema_cross_down'
            else 'none'
        end as signal_type
    from cte
)
select * from ema_cross_signals where signal_type != 'none'
""")

In [ ]:
rsi_signals_df = spark.sql(f"""
with cte as (
    select 
        symbol,
        interval,
        open_time,
        date,
        close_price,
        case
            when rsi6 < 30 then 'rsi_oversold'
            when rsi6 > 70 then 'rsi_overbought'
            else 'none'
        end as signal_type
    from {transform_db}.kline_features
)
select * from cte where signal_type != 'none'
""")

In [ ]:
macd_signals_df = spark.sql(f"""
with cte as (
    select 
        *,
        lag(macd) over(partition by symbol, interval order by open_time) as macd_prev,
        lag(signal) over(partition by symbol, interval order by open_time) as signal_prev
    from {transform_db}.kline_features
)
, macd_cross_signals as (
    select 
        symbol,
        interval,
        open_time,
        date,
        close_price,
        case
            when macd_prev is null or signal_prev is null then 'none'
            when macd_prev <= signal_prev and macd > signal then 'macd_cross_up'
            when macd_prev >= signal_prev and macd < signal then 'macd_cross_down'
            else 'none'
        end as signal_type
    from cte
)
select * from macd_cross_signals where signal_type != 'none'
""")

In [ ]:
patterns_df = spark.sql(f"""
with hammer as (
    select
        symbol,
        interval,
        open_time,
        date,
        close_price,
        'hammer' as signal_type
    from {transform_db}.kline_features
    where pattern_hammer is true
)
, bullish_engulfing as (
    select 
        symbol,
        interval,
        open_time,
        date,
        close_price,
        'bullish_engulfing' as signal_type
    from {transform_db}.kline_features
    where pattern_bullish_engulfing is true
)
, bearish_engulfing as (
    select 
        symbol,
        interval,
        open_time,
        date,
        close_price,
        'bearish_engulfing' as signal_type
    from {transform_db}.kline_features
    where pattern_bearish_engulfing is true
)
select * from hammer
union all
select * from bullish_engulfing
union all
select * from bearish_engulfing
""")

In [ ]:
from pyspark.sql import functions as F
signals_df = ema_signals_df.unionByName(rsi_signals_df).unionByName(macd_signals_df).unionByName(patterns_df) \
    .withColumnRenamed("open_time", "signal_time") \
    .withColumnRenamed("close_price", "entry_price") \
    .withColumn("signal_id", F.md5(F.concat_ws('_', F.col("symbol"), F.col("interval"), F.col("signal_time"), F.col("date"), F.col("signal_type")))) \
    .withColumn("signal_direction", F.when(F.col("signal_type").isin('ema_cross_up','rsi_oversold','macd_cross_up','bullish_engulfing','hammer'), 'bullish').otherwise('bearish'))

In [ ]:
signals_df.createOrReplaceTempView("signals_view")
spark.sql("""
select 
    *,
    CASE
        WHEN hour(signal_time) BETWEEN 0 AND 7 THEN 'asia'
        WHEN hour(signal_time) BETWEEN 8 AND 15 THEN 'europe'
        ELSE 'us'
    END AS session_name,
    dayofweek(signal_time) IN (1,7) AS is_weekend
from signals_view
""").show(20, False)

In [ ]:
spark.sql